In [1]:
from joblib import Parallel, delayed
import os
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm
pd.set_option('display.max_rows', 200)
pd.get_option('display.max_columns', 30) 

20

In [ ]:
# ss = pd.read_parquet(os.path.join(TRD_PATH, filter_files[0]))
# ss1 = pd.read_parquet(os.path.join(TRD_PATH, filter_files[1]))
TRD_PATH = "D:/guojun/okxapi/crypto_futures_data/selfdump/trades/"
start_date = '1762064220.9475596'
end_date = '1762065228.3686776'
filter_files = sorted([file for file in os.listdir(TRD_PATH) if ((file>=start_date) & (file<=end_date))])

from tqdm import tqdm
results = []
for file in tqdm(filter_files):
    ss = pd.read_parquet(os.path.join(TRD_PATH, file))
    results.append(ss[-3:])

df_trades = pd.concat(results)
df_trades.drop_duplicates(subset=['TIME', 'RIC', 'id'], inplace=True)
df_trades.to_csv('trade0.csv')

In [27]:
BOOK_PATH = "D:/guojun/okxapi/crypto_futures_data/selfdump/order_books/"
filter_files = sorted([file for file in os.listdir(BOOK_PATH) if ((file>=start_date) & (file<=end_date))])

from tqdm import tqdm
results = []
for file in tqdm(filter_files):
    ss = pd.read_parquet(os.path.join(BOOK_PATH, file))
    results.append(ss[-3:])
df_quote = pd.concat(results)
df_quote.drop_duplicates(subset=['TIME', 'RIC', 'timestart'], inplace=True)
df_quote.to_csv('quote.csv')

100%|██████████| 9001/9001 [00:57<00:00, 157.30it/s]


In [50]:
import pandas as pd
import numpy as np
# import swifter

def process_row(row):
    """处理单行数据的函数，用于并行化应用"""
    row_data = {
        'time': row['TIME'],
        'symbol': row['RIC'],
        'bid_price': np.nan,
        'ask_price': np.nan,
        'bid_size': np.nan,
        'ask_size': np.nan,
        'timestamp': row['timestart']
    }
    # 初始化所有目标列（填充NaN）
    target_columns = [
        'avg_price_-5', 'avg_price_-4', 'avg_price_-3', 'avg_price_-2',
        'avg_price_2', 'avg_price_3', 'avg_price_4', 'avg_price_5',
        'discrete_depth_-5', 'discrete_depth_-4', 'discrete_depth_-3',
        'discrete_depth_-2', 'discrete_depth_2', 'discrete_depth_3',
        'discrete_depth_4', 'discrete_depth_5',
        'discrete_notional_-5', 'discrete_notional_-4', 'discrete_notional_-3',
        'discrete_notional_-2', 'discrete_notional_-1', 'discrete_notional_1',
        'discrete_notional_2', 'discrete_notional_3', 'discrete_notional_4',
        'discrete_notional_5'
    ]
    row_data.update({col: np.nan for col in target_columns})
    
    # 处理bids（负向层级）
    bids = row['bids'] #if isinstance(row['bids'], list) else []
    # -1层级（最佳买价）
    row_data['bid_price'] = bids[0][0]
    row_data['discrete_notional_-1'] = bids[0][1] * bids[0][0] if bids[0][0] != 0 else np.nan
    row_data['bid_size'] = bids[0][1] 
    
    # -2到-5层级
    for i in range(0, 5):  # i=1对应-2，i=4对应-5
        level = -(i + 1)
        if i < len(bids):
            price, notional = bids[i][0], bids[i][1]
            row_data[f'avg_price_{level}'] = price
            row_data[f'discrete_notional_{level}'] =  notional * price if price != 0 else np.nan
            row_data[f'discrete_depth_{level}'] = notional
    
    # 处理asks（正向层级）
    asks = row['asks'] #if isinstance(row['asks'], list) else []
    # 1层级（最佳卖价）
    row_data['ask_price'] = asks[0][0]
    row_data['discrete_notional_1'] = asks[0][1] * asks[0][0] if asks[0][0] != 0 else np.nan
    row_data['ask_size'] = asks[0][1] 
    
    # 2到5层级
    for i in range(0, 5):  # i=1对应2，i=4对应5
        level = i + 1
        if i < len(asks):
            price, notional = asks[i][0], asks[i][1]
            row_data[f'avg_price_{level}'] = price
            row_data[f'discrete_notional_{level}'] =  notional * price if price != 0 else np.nan
            row_data[f'discrete_depth_{level}'] = notional
    
    return pd.Series(row_data)

# 使用swifter进行并行处理（自动利用多核）
quotes = df_quote.apply(process_row, axis=1)

In [49]:
trades = df_trades[['id', 'price', 'amount',
       'takerOrMaker', 'symbol', 'TIME', 'side', 'timestamp']].rename(columns={'price':'trade_price', 'amount':'trade_qty', 'TIME':'time', 'takerOrMaker':'is_buyer_maker'})

In [53]:
trades.to_csv('trades.csv')

In [149]:
def identify_order_events(quote_df, trade_df=None):
    """识别订单事件：挂单(AddOrder)、撤单(CancelOrder)和成交(Trade)"""
    # 确保数据按时间排序
    quote_df = quote_df.sort_values(['symbol', 'time']).reset_index(drop=True)
    
    # 初始化事件列表
    events = []
    
    # 按合约分组处理
    for symbol, group in quote_df.groupby('symbol'):
        # 所有档位列表
        bid_levels = [f'discrete_depth_-{i}' for i in range(1, 6)]
        ask_levels = [f'discrete_depth_{i}' for i in range(1, 6)]
        all_levels = bid_levels + ask_levels
        
        prev_row = None  # 保存前一行数据用于比较
        
        for _, row in tqdm(group.iterrows()):
            current_time = row['time']
            
            if prev_row is not None:
                # 比较每个档位的深度变化，识别挂单和撤单
                for level in all_levels:
                    # 提取价格列名
                    level_num = level.split('_')[-1]
                    if level_num == '-1':
                        price_col = 'bid_price'
                    elif level_num == '1':
                        price_col = 'ask_price'
                    else:
                        price_col = f'avg_price_{level_num}'
                    
                    prev_depth = prev_row[level]
                    curr_depth = row[level]
                    price = row[price_col]
                    
                    if curr_depth > prev_depth:
                        # 挂单事件
                        events.append({
                            'time': current_time,
                            'symbol': symbol,
                            'event_type': 'AddOrder',
                            'level': level,
                            'price': price,
                            'size_change': curr_depth - prev_depth,
                            'timestamp':row['timestamp'],
                            'bid_price':row['bid_price'],
                            'ask_price':row['ask_price'],
                            'bid_size':row['bid_size'],
                            'ask_size':row['ask_size']
                        })
                    elif curr_depth < prev_depth:
                        # 撤单事件
                        events.append({
                            'time': current_time,
                            'symbol': symbol,
                            'event_type': 'CancelOrder',
                            'level': level,
                            'price': price,
                            'size_change': curr_depth - prev_depth,
                            'timestamp':row['timestamp'],
                            'bid_price':row['bid_price'],
                            'ask_price':row['ask_price'],
                            'bid_size':row['bid_size'],
                            'ask_size':row['ask_size']
                        })
            
            prev_row = row
    
    # 转换为DataFrame
    events_df = pd.DataFrame(events)
    
    # 添加成交事件
    if trade_df is not None and not trade_df.empty:
        trade_events = trade_df.rename(columns={
            'trade_price': 'price',
            'trade_qty': 'size_change'
        })[['time', 'symbol', 'price', 'size_change']]
        trade_events['event_type'] = 'Trade'
        trade_events['level'] = trade_df['side']
        trade_events['timestamp'] = trade_df['timestamp']
        events_df = pd.concat([events_df, trade_events], ignore_index=True)
        events_df = events_df.sort_values(['symbol', 'time']).reset_index(drop=True)
    
    return events_df


In [150]:
events_df = identify_order_events(quotes, trades)

9003it [00:02, 4358.05it/s]


In [ ]:
events_df['mid_price'] = (events_df['bid_price'] + events_df['ask_price']) / 2
events_df.timestamp = pd.to_datetime(events_df.time, unit='s')

events_df['ts_1min'] = events_df.timestamp.dt.ceil('1min')
Omin_df = events_df.groupby(['ts_1min'])['mid_price'].last()

OminLag60s_df = Omin_df.shift(-1)
OminLag180s_df = Omin_df.shift(-3)
OminLag300s_df = Omin_df.shift(-5)
OminLag600s_df = Omin_df.shift(-10)
OminLag1800s_df = Omin_df.shift(-30)
OminLag3600s_df = Omin_df.shift(-60)
OminLag_df = pd.concat([OminLag60s_df, OminLag180s_df, OminLag300s_df, OminLag600s_df, OminLag1800s_df, OminLag3600s_df], axis=1)

OminLag_df.columns=[f'mid_price_lag{lag}s' for lag in [60, 180, 300, 600, 1800, 3600]]

df_merge = events_df.merge(OminLag_df, on=['ts_1min'], how='left')

for lag in [60, 180, 300, 600, 1800, 3600]:
    df_merge[f'LA{lag}'] = (df_merge[f'mid_price_lag{lag}s'] / df_merge['mid_price'] - 1.) * 10000
events_df = df_merge

In [94]:
# import pandas as pd

# def mark_trade_del(event_df):
#     df = event_df.copy()
#     trade_indices = df[df['event_type'] == 'Trade'].index.tolist()
    
#     for idx in tqdm(trade_indices):
#         trade_price = df.at[idx, 'price']
#         bid_level = df[df['level'] == 'discrete_depth_-1']['price'].max() if not df[df['level'] == 'discrete_depth_-1'].empty else None
#         ask_level = df[df['level'] == 'discrete_depth_1']['price'].min() if not df[df['level'] == 'discrete_depth_1'].empty else None
        
#         side = df.at[idx, 'level']
        
#         # 确定目标层级
#         target_level = 'discrete_depth_1' if side=='buy' else 'discrete_depth_-1'
        
#         end_idx = min(idx + 21, len(df))  
#         look_window = df.iloc[idx+1:end_idx]
        
#         mask = (look_window['price'] == trade_price) & (look_window['level'] == target_level)
#         target_rows = look_window[mask]
        
#         if not target_rows.empty:
#             target_idx = target_rows.index[0]
#             # 修改事件类型为TradeDel
#             df.at[target_idx, 'event_type'] = 'TradeDel'
    
#     return df

# # 使用示例：
# # 假设你的数据框名为event_df
# modified_df0 = mark_trade_del(events_df)
# modified_df0.to_csv('modified_df0.csv')

100%|██████████| 2466/2466 [01:49<00:00, 22.49it/s]


In [152]:
import pandas as pd
import numpy as np

def mark_trade_del_vectorized(event_df):
    # 复制数据并按时间戳排序（确保事件顺序）
    df = event_df.copy().sort_values('timestamp').reset_index(drop=True)
    
    # 1. 识别所有 Trade 事件的索引和属性
    trade_mask = df['event_type'] == 'Trade'
    if not trade_mask.any():
        return df  # 无 Trade 事件直接返回
    
    # 提取 Trade 事件的关键信息（索引、价格、方向）
    trade_indices = df.index[trade_mask].to_numpy()
    trade_prices = df.loc[trade_mask, 'price'].to_numpy()
    
    # 假设 Trade 事件有 'side' 列直接标记买卖方向（若没有需调整判断逻辑）
    # 这里使用 'side' 列作为示例，若实际数据无此列，可参考上一版的盘口推断逻辑
    is_buy = df.loc[trade_mask, 'level'] == 'buy'
    target_levels = np.where(is_buy, 'discrete_depth_1', 'discrete_depth_-1')
    
    # 2. 为每个行标记其属于哪个 Trade 的搜索窗口（后续 20 行）
    # 创建窗口标记数组：每行属于哪个 Trade 的搜索范围（-1 表示不属于任何窗口）
    window_marker = np.full(len(df), -1, dtype=int)
    
    for i, trade_idx in tqdm(enumerate(trade_indices)):
        # 计算当前 Trade 事件的搜索窗口范围 [trade_idx+1, min(trade_idx+20, len(df)-1)]
        start = trade_idx + 1
        end = min(trade_idx + 20, len(df) - 1)
        if start > end:
            continue  # 窗口无效（超出数据范围）
        window_marker[start:end+1] = i  # 标记窗口内的行属于第 i 个 Trade
    
    # 3. 向量化匹配：找出所有在 Trade 窗口内且符合条件的行
    # 将窗口标记、价格、层级转为数组便于向量化操作
    prices = df['price'].to_numpy()
    levels = df['level'].to_numpy()
    
    # 条件1：在某个 Trade 的搜索窗口内（window_marker != -1）
    # 条件2：价格与对应 Trade 的价格一致
    # 条件3：层级与对应 Trade 的目标层级一致
    mask = (window_marker != -1) & \
           (prices == trade_prices[window_marker]) & \
           (levels == target_levels[window_marker])
    
    # 4. 去重：每个 Trade 只保留第一个匹配的行
    # 对符合条件的行按 Trade 分组，取每组第一个索引
    valid_indices = df.index[mask]
    if len(valid_indices) == 0:
        return df  # 无匹配行直接返回
    
    # 按窗口标记分组，取每组第一个索引
    first_matches = (
        pd.DataFrame({
            'window_id': window_marker[mask],
            'index': valid_indices
        })
        .groupby('window_id')['index'].first()
        .values
    )
    
    # 5. 修改事件类型为 TradeDel
    df.loc[first_matches, 'event_type'] = 'TradeDel'
    
    return df
modified_df = mark_trade_del_vectorized(events_df)

2466it [00:00, 144087.17it/s]


In [271]:
modified_df0 = pd.read_csv('modified_df.csv', index_col=0)

In [272]:
import pandas as pd
import numpy as np

# 假设你已完成的代码如下
modified_df_add = modified_df0[modified_df0.event_type=='AddOrder']
modified_df_add['level_BS'] = modified_df_add['level'].apply(lambda x: 'bid' if int(x.split('_')[-1])<0 else 'ask')
modified_df_add['TIME_BIN'] = np.ceil(modified_df_add['time'] / 60) * 60
modified_df_add['TIME_BIN'] = (modified_df_add['TIME_BIN'] * 10**6).astype('int64')
bin_sameadd_vol = modified_df_add.groupby(['TIME_BIN', 'symbol', 'level_BS', 'price'])['size_change'].sum().sort_index()


# ---------------------- 任务1：计算每1分钟内最大AddOrder档位距离mid价格的ticks数 ----------------------
# 1. 计算每分钟的最佳买卖价（best bid/ask）
# 分离bid和ask数据
bid_mask = bin_sameadd_vol.index.get_level_values('level_BS') == 'bid'
ask_mask = bin_sameadd_vol.index.get_level_values('level_BS') == 'ask'

# 计算每分钟最佳bid（bid中最高价格）和最佳ask（ask中最低价格）
best_bid_min = bin_sameadd_vol[bid_mask].groupby(level=['TIME_BIN', 'symbol']).apply(
    lambda x: x.index.get_level_values('price').max() if not x.empty else np.nan
)
best_ask_min = bin_sameadd_vol[ask_mask].groupby(level=['TIME_BIN', 'symbol']).apply(
    lambda x: x.index.get_level_values('price').min() if not x.empty else np.nan
)

# 计算mid价格（最佳买卖价的平均）
mid_prices_min = (best_bid_min + best_ask_min) / 2

# 2. 找到每分钟AddOrder size最大的档位（峰档位）
max_size_idx_min = bin_sameadd_vol.groupby(level=['TIME_BIN', 'symbol', 'level_BS']).idxmax()  # 峰档位完整索引
peak_prices_min = pd.Series(
    [idx[3] for idx in max_size_idx_min],  # 提取峰档位价格
    index=max_size_idx_min.index,
    name='peak_price'
)

# 3. 计算距离mid价格的ticks数（假设ticksize=0.01，可根据实际调整）
ticksize = 0.01  # 最小价格变动单位
task1 = (peak_prices_min - mid_prices_min).abs() / ticksize
task1 = task1.rename('peak_to_mid_ticks_per_min')


# ---------------------- 修正任务2和3的代码 ----------------------
# 1. 构建5分钟时间桶（保持不变）
modified_df_add['TIME_BIN_5min'] = modified_df_add['TIME_BIN']

# 2. 计算5分钟内各档位的总AddOrder量（保持不变）
bin_sameadd_vol_5min = modified_df_add.groupby(
    ['TIME_BIN_5min', 'symbol', 'level_BS', 'price']
)['size_change'].sum().sort_index()

# 3. 计算5分钟内总AddOrder量（保持不变）
total_add_5min_B = bin_sameadd_vol_5min[bin_sameadd_vol_5min.index.get_level_values('level_BS') == 'bid'].groupby(
    level=['TIME_BIN_5min', 'symbol']
).sum().rename('total_add_size_5min')
total_add_5min_S = bin_sameadd_vol_5min[bin_sameadd_vol_5min.index.get_level_values('level_BS') == 'ask'].groupby(
    level=['TIME_BIN_5min', 'symbol']
).sum().rename('total_add_size_5min')

# 4. 确定5分钟内的峰档位（size最大的档位）（保持不变）
peak_idx_5min = bin_sameadd_vol_5min.groupby(
    level=['TIME_BIN_5min', 'symbol', 'level_BS']
).idxmax()  # 峰档位完整索引
peak_size_5min = bin_sameadd_vol_5min.loc[peak_idx_5min].rename('peak_size_5min')

# 5. 计算5分钟内的mid价格（保持不变）
best_bid_5min = bin_sameadd_vol_5min[
    bin_sameadd_vol_5min.index.get_level_values('level_BS') == 'bid'
].groupby(level=['TIME_BIN_5min', 'symbol']).apply(
    lambda x: x.index.get_level_values('price').max() if not x.empty else np.nan
)
best_ask_5min = bin_sameadd_vol_5min[
    bin_sameadd_vol_5min.index.get_level_values('level_BS') == 'ask'
].groupby(level=['TIME_BIN_5min', 'symbol']).apply(
    lambda x: x.index.get_level_values('price').min() if not x.empty else np.nan
)
mid_prices_5min = ((best_bid_5min + best_ask_5min) / 2).rename('mid_price_5min')

# 6. 修正：计算比峰档位更靠近mid的AddOrder量占比（任务2）
def get_closer_size(group):
    """计算组内比峰档位更靠近mid的总size"""
    time_bin, symbol, level_BS = group.name  # group的分组键是(TIME_BIN_5min, symbol)
    peak_idx = peak_idx_5min.loc[(time_bin, symbol, level_BS)]  # 获取当前组的峰档位索引
    if pd.isna(peak_idx):  # 处理无峰档位的情况
        return 0
    
    peak_price = peak_idx[3]  # 峰档位的价格（索引第3层是price）
    mid_price = mid_prices_5min.loc[(time_bin, symbol)]  # 当前组的mid价格
    
    if pd.isna(mid_price):  # 处理无mid价格的情况
        return 0
    
    # 峰档位到mid的距离
    peak_dist = abs(peak_price - mid_price)
    
    # 关键修正：直接用数组计算布尔掩码（不转换为Series，避免索引问题）
    # 提取当前组所有档位的价格
    prices = group.index.get_level_values('price').to_numpy()
    # 生成与group长度一致的布尔数组（不包含索引）
    closer_mask = np.abs(prices - mid_price) < peak_dist
    
    # 用布尔数组筛选并求和
    return group[closer_mask].sum()

# 修正分组键：将'TIME_BIN'改为'TIME_BIN_5min'（原代码笔误）
closer_size_5min = bin_sameadd_vol_5min.groupby(
    level=['TIME_BIN_5min', 'symbol', 'level_BS']  # 这里之前写错了，应该是5分钟桶
).apply(get_closer_size).rename('closer_size_5min')


closer_size_5min_ = pd.DataFrame(closer_size_5min).unstack('level_BS')
closer_size_5min_.columns = ['size_after_peak_ask', 'size_after_peak_bid']
closer_size_5min_['percent_after_peak_ask'] = closer_size_5min_['size_after_peak_ask'] / total_add_5min_S
closer_size_5min_['percent_after_peak_bid'] = closer_size_5min_['size_after_peak_bid'] / total_add_5min_B

peak_size_5min_ = pd.DataFrame(peak_size_5min).reset_index('price', drop=True).unstack('level_BS')
peak_size_5min_.columns = ['peak_size_ask', 'peak_size_bid']
peak_size_5min_['percent_peak_ask'] = peak_size_5min_['peak_size_ask'] / total_add_5min_S
peak_size_5min_['percent_peak_bid'] = peak_size_5min_['peak_size_bid'] / total_add_5min_B

peak_price_5min_ = pd.DataFrame(peak_size_5min).reset_index('price')['price'].unstack('level_BS')
peak_price_5min_.columns = ['peak_price_ask', 'peak_price_bid']

task1_ = pd.DataFrame(task1).unstack('level_BS')
task1_.columns = ['ticksize_pick_mid_ask', 'ticksize_pick_mid_bid']
results_concat_df = pd.concat([task1_, peak_price_5min_, peak_size_5min_, closer_size_5min_], axis=1)


C:\Users\HOMON\AppData\Local\Temp\ipykernel_24172\3677988802.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  modified_df_add['level_BS'] = modified_df_add['level'].apply(lambda x: 'bid' if int(x.split('_')[-1])<0 else 'ask')
C:\Users\HOMON\AppData\Local\Temp\ipykernel_24172\3677988802.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  modified_df_add['TIME_BIN'] = np.ceil(modified_df_add['time'] / 60) * 60
C:\Users\HOMON\AppData\Local\Temp\ipykernel_24172\3677988802.py:8: SettingWithCopyWarning: 
A val

In [273]:
results_concat_df

,,ticksize_pick_mid_ask,ticksize_pick_mid_bid,peak_price_ask,peak_price_bid,peak_size_ask,peak_size_bid,percent_peak_ask,percent_peak_bid,size_after_peak_ask,size_after_peak_bid,percent_after_peak_ask,percent_after_peak_bid
,symbol,,,,,,,,,,,,
1762064280000000,SOL/USDT:USDT,5.000000e-01,4.5,186.61,186.56,2847.89,3339.19,0.168267,0.163128,860.23,14317.84,0.050826,0.699462
1762064340000000,SOL/USDT:USDT,8.500000e+00,3.5,186.77,186.72,1492.17,1898.41,0.093790,0.095479,9873.62,4397.73,0.620603,0.221179
1762064400000000,SOL/USDT:USDT,2.000000e+00,3.0,186.83,186.78,1483.17,2655.85,0.085977,0.115275,1780.25,5872.78,0.103199,0.254903
1762064460000000,SOL/USDT:USDT,1.500000e+00,5.5,186.88,186.81,2827.56,3609.54,0.149156,0.161759,1404.78,13530.31,0.074103,0.606350
1762064520000000,SOL/USDT:USDT,4.500000e+00,11.5,186.69,186.62,1443.03,2669.17,0.108756,0.138879,5940.73,14263.97,0.447732,0.742163
1762064580000000,SOL/USDT:USDT,1.500000e+00,1.5,186.75,186.78,2222.10,3718.25,0.109283,0.129610,1474.05,4981.16,0.072494,0.173632
1762064640000000,SOL/USDT:USDT,5.000000e+00,7.0,186.63,186.61,1612.04,3429.91,0.093067,0.127578,7545.80,7364.62,0.435637,0.273932
1762064700000000,SOL/USDT:USDT,9.000000e+00,0.0,186.73,186.64,1741.01,1660.73,0.112367,0.098399,12733.06,0.00,0.821805,0.000000
1762064760000000,SOL/USDT:USDT,4.000000e+00,6.0,186.60,186.50,3513.86,9050.62,0.240740,0.423440,8916.92,11627.99,0.610913,0.544024


In [306]:
for col in ['ticksize_pick_mid', 'peak_size', 'percent_peak', 'percent_after_peak']:
    results_concat_df[f'{col}_BSdiff'] = results_concat_df[f'{col}_bid'] - results_concat_df[f'{col}_ask'] 
    results_concat_df[f'{col}_BSimb'] = (results_concat_df[f'{col}_bid'] - results_concat_df[f'{col}_ask'] ) / (results_concat_df[f'{col}_bid'] + results_concat_df[f'{col}_ask'] )
    results_concat_df[f'{col}_BSlog'] = np.log( 1 + results_concat_df[f'{col}_bid'] ) / (1 + results_concat_df[f'{col}_ask'] )

In [308]:

events_df_1min = events_df.groupby(['symbol', 'TIME_BIN'])[['LA60', 'LA180', 'LA300', 'LA600', 'LA1800',
       'LA3600', 'bid_price', 'ask_price', 'bid_size', 'ask_size']].last().reset_index()
results_withlag = results_concat_df.merge(events_df_1min, left_on=['level_0', 'symbol'], right_on = ['TIME_BIN', 'symbol'], how='left')



In [ ]:
def plot_cols(feat_col, df):
    # # 假设你已经有以下变量
    # feat_col = 'Around3s_AvgPrc_minusclose'
    ret_cols = ['ret5m', 'ret30m']

    # ---------- 全部时间点 (FR_all_*) ----------
    fr_all_results = {}
    for ret_col in ret_cols:
        sub_df = df[[feat_col, ret_col, 'date']].dropna()

        # 按date groupby，计算 FR = cov(x, y) / std(x)
        grouped = sub_df.groupby('date')
        mean_x = grouped[feat_col].transform('mean')
        mean_y = grouped[ret_col].transform('mean')
        std_x = grouped[feat_col].transform('std')

        cov_xy = ((sub_df[feat_col] - mean_x) * (sub_df[ret_col] - mean_y)).groupby(sub_df['date']).mean()
        std_x_daily = std_x.groupby(sub_df['date']).first()  # std(x)是一样的，随便取一个就行

        fr = cov_xy / std_x_daily.replace(0, np.nan)
        fr_all_results[f'FR_all_{ret_col}'] = fr

    # 合并所有结果
    fr_df = pd.DataFrame(fr_all_results)

    # 计算 cumulative FR
    cumulative_fr_df = fr_df.cumsum()
    # 分别提取 ret0e 和 ret1h 的 FR 列
    fr_0e_cols = [col for col in cumulative_fr_df.columns if 'ret5m' in col]
    fr_1h_cols = [col for col in cumulative_fr_df.columns if 'ret30m' in col]

    # 画 ret0e_shift 的 Cumulative FR 图
    plt.figure(figsize=(16, 6))
    for col in fr_0e_cols:
        plt.plot(cumulative_fr_df.index, cumulative_fr_df[col], label=col)

    plt.title(f'{feat_col} Cumulative FR (ret5m)')
    plt.xlabel('Date')
    plt.ylabel('Cumulative FR')
    plt.legend()
    plt.grid(True)
    plt.xticks(cumulative_fr_df.index[::10], rotation=45)
    plt.tight_layout()
    plt.show()

    # 画 ret1h_shift 的 Cumulative FR 图
    plt.figure(figsize=(16, 6))
    for col in fr_1h_cols:
        plt.plot(cumulative_fr_df.index, cumulative_fr_df[col], label=col)

    plt.title(f'{feat_col} Cumulative FR (ret30m) -|')
    plt.xlabel('Date')
    plt.ylabel('Cumulative FR')
    plt.legend()
    plt.grid(True)
    plt.xticks(cumulative_fr_df.index[::10], rotation=45)
    plt.tight_layout()
    plt.show()
#     fr_df.to_parquet(f'/data/beef3/mike/pshared/GAlpha_custom_build/wensheng_jawN9J/fr_save/{feat_col}.parquet', engine="pyarrow", compression="snappy")
    

plot_cols('Signal', results_withlag)

,level_0,symbol,ticksize_pick_mid_ask,ticksize_pick_mid_bid,peak_price_ask,peak_price_bid,peak_size_ask,peak_size_bid,percent_peak_ask,percent_peak_bid,size_after_peak_ask,size_after_peak_bid,percent_after_peak_ask,percent_after_peak_bid,ticksize_pick_mid_BSdiff,ticksize_pick_mid_BSimb,ticksize_pick_mid_BSlog
0,1762064280000000,SOL/USDT:USDT,5.000000e-01,4.5,186.61,186.56,2847.89,3339.19,0.168267,0.163128,860.23,14317.84,0.050826,0.699462,4.000000e+00,8.000000e-01,1.136499
1,1762064340000000,SOL/USDT:USDT,8.500000e+00,3.5,186.77,186.72,1492.17,1898.41,0.093790,0.095479,9873.62,4397.73,0.620603,0.221179,-5.000000e+00,-4.166667e-01,0.158324
2,1762064400000000,SOL/USDT:USDT,2.000000e+00,3.0,186.83,186.78,1483.17,2655.85,0.085977,0.115275,1780.25,5872.78,0.103199,0.254903,1.000000e+00,2.000000e-01,0.462098
3,1762064460000000,SOL/USDT:USDT,1.500000e+00,5.5,186.88,186.81,2827.56,3609.54,0.149156,0.161759,1404.78,13530.31,0.074103,0.606350,4.000000e+00,5.714286e-01,0.748721
4,1762064520000000,SOL/USDT:USDT,4.500000e+00,11.5,186.69,186.62,1443.03,2669.17,0.108756,0.138879,5940.73,14263.97,0.447732,0.742163,7.000000e+00,4.375000e-01,0.459223
5,1762064580000000,SOL/USDT:USDT,1.500000e+00,1.5,186.75,186.78,2222.10,3718.25,0.109283,0.129610,1474.05,4981.16,0.072494,0.173632,2.842171e-12,9.473903e-13,0.366516
6,1762064640000000,SOL/USDT:USDT,5.000000e+00,7.0,186.63,186.61,1612.04,3429.91,0.093067,0.127578,7545.80,7364.62,0.435637,0.273932,2.000000e+00,1.666667e-01,0.346574
7,1762064700000000,SOL/USDT:USDT,9.000000e+00,0.0,186.73,186.64,1741.01,1660.73,0.112367,0.098399,12733.06,0.00,0.821805,0.000000,-9.000000e+00,-1.000000e+00,0.000000
8,1762064760000000,SOL/USDT:USDT,4.000000e+00,6.0,186.60,186.50,3513.86,9050.62,0.240740,0.423440,8916.92,11627.99,0.610913,0.544024,2.000000e+00,2.000000e-01,0.389182
9,1762064820000000,SOL/USDT:USDT,5.500000e+00,1.5,186.68,186.61,3062.13,3036.34,0.247035,0.153643,6076.02,4438.39,0.490178,0.224589,-4.000000e+00,-5.714286e-01,0.140968


In [ ]:
# 挂单的筹码分布skew 时间做一个decay，偏度的变化量，（全量订单簿变化量的偏度， 最后一个时刻的偏度）

# 挂单的筹码：价格上涨：新增档位委买量+原档位净委买增加量+主买成交额； 被动卖出成交额+原档位净委卖减少量

# 新增档位委买量 / 被动卖出成交额；
# （新增档位委买量 + 净主买成交额） /  净被动卖出成交额

# （新增档位委买量 + 原档位净委买增加量） / （被动卖出成交额 + 原档位净委卖减少量）

# （新增档位委买量 + 原档位净委买增加量 + 净主买成交额） / （净被动卖出成交额 + 原档位净委卖减少量）

# 全量订单簿变化量

